# 테이블 생성/데이터 입력 노트북

이 노트북은 다음 순서로 사용합니다.
1. `TABLE_NAME`, `SCHEMA`를 확인
2. 테이블 생성 셀 실행
3. `TEST_ID`, `ID_PREFIX`, `RAW_ROWS`를 작성
4. 업서트 셀 실행
5. 결과 조회 셀 실행


# normcondition 데이터

`normcondition`은 검사별 규준 조건 마스터입니다.
현재는 `parent_item` 또는 `parent_scale`의 고유 `sub_test_json` 조건을 기준으로 1건씩 생성할 수 있습니다.

`id`는 직접 쓰지 않고 `ID_PREFIX` + `id_suffix` 조합으로 자동 생성합니다.
예: `STS_NORMCOND_infant`, `STS_NORMCOND_adult`


In [1]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

# DB 경로: 검사 설계용 별도 DB
DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\modular.db")
TABLE_NAME = 'normcondition'

# type: INTEGER(정수) / REAL(실수) / TEXT / BLOB(바이너리) 중 하나 사용
# constraints 예시: 'PRIMARY KEY AUTOINCREMENT', 'NOT NULL', 'DEFAULT 0'
SCHEMA = [
    {'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'test.id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'type', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'condition', 'type': 'TEXT', 'constraints': 'NOT NULL'},
]
PRIMARY_KEY = ['id', 'test.id']

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_\.]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어/점만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)


설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\modular.db
TABLE_NAME = normcondition
SCHEMA = [{'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'test.id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'type', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'condition', 'type': 'TEXT', 'constraints': 'NOT NULL'}]
PRIMARY_KEY = ['id', 'test.id']


In [2]:
# 2) 테이블 생성
column_defs = []
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').strip()
    part = f'"{c_name}" {c_type}'
    if c_constraints:
        part += f' {c_constraints}'
    column_defs.append(part)

if PRIMARY_KEY:
    pk_cols = ', '.join([f'"{c}"' for c in PRIMARY_KEY])
    column_defs.append(f'PRIMARY KEY ({pk_cols})')

create_sql = f'CREATE TABLE IF NOT EXISTS "{TABLE_NAME}" (' + ', '.join(column_defs) + ')'
print('실행 SQL:', create_sql)

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(create_sql)
    conn.commit()
    print(f'테이블 생성 완료: {TABLE_NAME}')
finally:
    conn.close()


실행 SQL: CREATE TABLE IF NOT EXISTS "normcondition" ("id" TEXT NOT NULL, "test.id" TEXT NOT NULL, "name" TEXT NOT NULL, "type" TEXT NOT NULL, "condition" TEXT NOT NULL, PRIMARY KEY ("id", "test.id"))
테이블 생성 완료: normcondition


## 업서트


In [4]:
# 3) normcondition 테이블 업서트
import json

# 검사별 공통 prefix를 먼저 정합니다.
TEST_ID = 'PET'

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 id 대신 id_suffix를 작성합니다.
# 최종 id는 f'{TEST_ID}_{id_suffix}' 형태로 자동 생성됩니다.
RAW_ROWS = [
    {
        'id_suffix': 'ALL',
        'name': '모두(연령확대)',
        'type': 'age_range',
        'condition': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [0, 0, 0],
                'end_exclusive': [20, 0, 0],
            },
            'gender': ['male', 'female'],
        },
    },
]

def build_id(id_prefix: str, id_suffix: str) -> str:
    suffix = str(id_suffix).strip()
    if not suffix:
        raise ValueError('id_suffix는 비어 있을 수 없습니다.')
    return f'{id_prefix}_{suffix}'

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_raw_keys = {'id_suffix', 'name', 'type', 'condition'}
normalized_rows = []
for i, row in enumerate(RAW_ROWS, start=1):
    if set(row.keys()) != expected_raw_keys:
        raise ValueError(
            f'{i}번째 RAW_ROW 키 불일치: expected={sorted(expected_raw_keys)}, got={sorted(row.keys())}'
        )

    materialized = {
        'id': build_id(TEST_ID, row['id_suffix']),
        'test.id': TEST_ID,
        'name': row['name'],
        'type': row['type'],
        'condition': row['condition'],
    }
    normalized = {k: normalize_value(k, materialized[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'"{c}"' for c in insert_columns])

# 기존 데이터를 덮어쓰도록 ON CONFLICT 절 추가 (sqlite)
conflict_columns = ['id', 'test.id']
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = (
    f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) '
    f'ON CONFLICT ({conflict_sql}) DO UPDATE SET {update_sql}'
)
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

print('실행 SQL:', insert_sql)
print('입력 건수:', len(values))
print('생성될 id 목록:', [row['id'] for row in normalized_rows])

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'업서트 완료: {len(values)}건')
finally:
    conn.close()


입력 대상 컬럼: ['id', 'test.id', 'name', 'type', 'condition']
실행 SQL: INSERT INTO "normcondition" ("id", "test.id", "name", "type", "condition") VALUES (?, ?, ?, ?, ?) ON CONFLICT ("id", "test.id") DO UPDATE SET "name" = excluded."name", "type" = excluded."type", "condition" = excluded."condition"
입력 건수: 1
생성될 id 목록: ['PET_ALL']
업서트 완료: 1건


In [5]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(
        f'SELECT * FROM "{TABLE_NAME}" WHERE "test.id" = ? ORDER BY "id"',
        (TEST_ID,),
    )
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()


총 1건
('PET_ALL', 'PET', '모두(연령확대)', 'age_range', '{"age_range": {"as_of_time": "00:00:00", "start_inclusive": [0, 0, 0], "end_exclusive": [20, 0, 0]}, "gender": ["male", "female"]}')
